# 🏆 Notebook 04 — Version Finale & Rapport Complet
**Projet :** Cervical Cancer Risk Prediction  
**Auteure :** Hadil Dhaya · 4th Year Data Science · Group 5  
**Dataset :** UCI — Cervical Cancer Risk Factors

---
Ce notebook est la **version de présentation finale** :  
il contient uniquement le meilleur modèle, les résultats parfaits,  
l'explainability SHAP complète et le rapport de conclusions.

**Prérequis :** avoir exécuté les notebooks 01, 02, 03 dans l'ordre.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
import shap
import os
import warnings

from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    accuracy_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score
)
from sklearn.model_selection import cross_val_score, StratifiedKFold

warnings.filterwarnings('ignore')
shap.initjs()
os.makedirs('../outputs', exist_ok=True)
print('✅ Imports OK')

## 1. Chargement du meilleur modèle et des données

In [ ]:
# Charger données test
X_train = pd.read_csv('../outputs/X_train.csv')
X_test  = pd.read_csv('../outputs/X_test.csv')
y_train = pd.read_csv('../outputs/y_train.csv').squeeze()
y_test  = pd.read_csv('../outputs/y_test.csv').squeeze()

# Charger le meilleur modèle (tuné)
best_model = joblib.load('../outputs/best_model_tuned.pkl')
best_name  = joblib.load('../outputs/best_model_name.pkl')

print(f'✅ Modèle chargé : {best_name}')
print(f'   X_test shape  : {X_test.shape}')
print(f'   y_test dist   : {dict(y_test.value_counts())}')

## 2. Performances finales du meilleur modèle

In [ ]:
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]
cm     = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

auc  = roc_auc_score(y_test, y_prob)
f1   = f1_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)
prec = precision_score(y_test, y_pred, zero_division=0)
acc  = accuracy_score(y_test, y_pred)

print('=' * 60)
print(f'  PERFORMANCES FINALES — {best_name}')
print('=' * 60)
print(f'  AUC-ROC   : {auc:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print(f'  Recall    : {rec:.4f}   ← métrique principale')
print(f'  Precision : {prec:.4f}')
print(f'  Accuracy  : {acc:.4f}')
print('=' * 60)
print(f'  TP={tp}  TN={tn}  FP={fp}  FN={fn}')
print(f'  Sensibilité : {tp/(tp+fn):.4f}')
print(f'  Spécificité : {tn/(tn+fp):.4f}')
print('=' * 60)

## 3. Cross-validation finale (robustesse)

In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_scores = cross_val_score(best_model, X_train, y_train,
                             cv=cv, scoring='roc_auc', n_jobs=-1)
print(f'Cross-validation 10-fold AUC :')
for i, s in enumerate(cv_scores):
    print(f'  Fold {i+1:2d} : {s:.4f}')
print(f'  ─────────────────')
print(f'  Mean ± Std : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'  Min / Max  : {cv_scores.min():.4f} / {cv_scores.max():.4f}')

## 4. Visualisation finale — Figure de présentation

In [ ]:
fpr_c, tpr_c, _ = roc_curve(y_test, y_prob)
prec_c, rec_c, _ = precision_recall_curve(y_test, y_prob)

fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.35)

# 1. ROC Curve
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(fpr_c, tpr_c, color='#2980b9', lw=2.5, label=f'AUC = {auc:.4f}')
ax1.fill_between(fpr_c, tpr_c, alpha=0.1, color='#2980b9')
ax1.plot([0,1],[0,1],'k--', lw=0.8)
ax1.set_xlabel('FPR'); ax1.set_ylabel('TPR')
ax1.set_title('ROC Curve', fontweight='bold')
ax1.legend(); ax1.grid(alpha=0.3)

# 2. Precision-Recall
ax2 = fig.add_subplot(gs[0, 1])
ap  = average_precision_score(y_test, y_prob)
ax2.plot(rec_c, prec_c, color='#27ae60', lw=2.5, label=f'AP = {ap:.4f}')
ax2.fill_between(rec_c, prec_c, alpha=0.1, color='#27ae60')
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve', fontweight='bold')
ax2.legend(); ax2.grid(alpha=0.3)

# 3. Confusion Matrix
ax3 = fig.add_subplot(gs[0, 2])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax3,
            xticklabels=['No Cancer','Cancer'],
            yticklabels=['No Cancer','Cancer'],
            linewidths=1.5, linecolor='white',
            annot_kws={'size':16,'weight':'bold'})
ax3.set_xlabel('Prédit'); ax3.set_ylabel('Réel')
ax3.set_title('Confusion Matrix', fontweight='bold')

# 4. Métriques barplot
ax4 = fig.add_subplot(gs[1, 0])
met_names = ['AUC-ROC','F1','Recall','Precision','Accuracy']
met_vals  = [auc, f1, rec, prec, acc]
bar_colors= ['#2980b9','#8e44ad','#e74c3c','#f39c12','#27ae60']
bars = ax4.bar(met_names, met_vals, color=bar_colors, edgecolor='white', width=0.6)
for bar, v in zip(bars, met_vals):
    ax4.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
             f'{v:.3f}', ha='center', fontweight='bold', fontsize=10)
ax4.set_ylim(0, 1.15); ax4.set_ylabel('Score')
ax4.set_title('Métriques finales', fontweight='bold')
ax4.grid(axis='y', alpha=0.3)

# 5. Cross-val distribution
ax5 = fig.add_subplot(gs[1, 1])
ax5.boxplot(cv_scores, vert=True, patch_artist=True,
            boxprops=dict(facecolor='#3498db', alpha=0.7),
            medianprops=dict(color='red', linewidth=2))
ax5.scatter([1]*len(cv_scores), cv_scores, color='#2c3e50', zorder=5, s=40)
ax5.set_ylabel('AUC-ROC')
ax5.set_title(f'CV 10-fold AUC\n{cv_scores.mean():.4f} ± {cv_scores.std():.4f}',
              fontweight='bold')
ax5.set_xticklabels([])
ax5.grid(axis='y', alpha=0.3)

# 6. Analyse FN/FP
ax6 = fig.add_subplot(gs[1, 2])
cats = ['VP\n(cancers\ndétectés)','VN\n(sains\ncorrects)',
        'FP\n(fausses\nalarmes)','FN\n(cancers\nmanqués)']
vals = [tp, tn, fp, fn]
clrs = ['#27ae60','#3498db','#f39c12','#e74c3c']
bars2 = ax6.bar(cats, vals, color=clrs, edgecolor='white', width=0.6)
for bar, v in zip(bars2, vals):
    ax6.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
             str(v), ha='center', fontweight='bold', fontsize=11)
ax6.set_ylabel('Nombre de patients')
ax6.set_title('Analyse des erreurs critiques', fontweight='bold')
ax6.grid(axis='y', alpha=0.3)

plt.suptitle(f'Résultats Finaux — {best_name} | Cervical Cancer Risk Prediction',
             fontsize=15, fontweight='bold', y=1.01)
plt.savefig('../outputs/final_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Sauvegardé : outputs/final_results.png')

## 5. SHAP — Explainability complète

In [ ]:
print('⏳ Calcul des SHAP values...')
try:
    explainer   = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_test)
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values
    ev = explainer.expected_value[1] if isinstance(explainer.expected_value, list) \
         else explainer.expected_value
    print('✅ TreeExplainer OK')
except Exception:
    print('TreeExplainer non compatible, passage à KernelExplainer (plus lent)...')
    bg = shap.sample(X_train, 50)
    explainer   = shap.KernelExplainer(best_model.predict_proba, bg)
    shap_values = explainer.shap_values(X_test.iloc[:80])
    sv = shap_values[1]
    ev = explainer.expected_value[1]
    print('✅ KernelExplainer OK')

In [ ]:
# 5a. Summary Plot (importance globale)
plt.figure(figsize=(10, 8))
# Extract SHAP values for class 1 (cancer) - handle both 2D and 3D arrays
if shap_values.ndim == 3:
    # Shape is (n_samples, n_features, 2)
    sv_2d = shap_values[:, :, 1]  # Extract class 1 for all samples
    X_subset = X_test.iloc[:len(sv_2d)]
else:
    sv_2d = sv
    X_subset = X_test.iloc[:len(sv_2d)]
shap.summary_plot(sv_2d, X_subset, feature_names=X_test.columns.tolist(),
                  plot_type='bar', show=False, color='#2980b9')
plt.title('SHAP — Feature Importance Globale', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 5b. Beeswarm Plot (direction + magnitude)
plt.figure(figsize=(10, 8))
# Extract SHAP values for class 1 (cancer) - handle both 2D and 3D arrays
if shap_values.ndim == 3:
	# Shape is (n_samples, n_features, 2)
	sv_beeswarm = shap_values[:, :, 1]  # Extract class 1 for all samples
	X_subset_beeswarm = X_test.iloc[:len(sv_beeswarm)]
else:
	sv_beeswarm = sv
	X_subset_beeswarm = X_test.iloc[:len(sv_beeswarm)]
shap.summary_plot(sv_beeswarm, X_subset_beeswarm, feature_names=X_test.columns.tolist(), show=False)
plt.title('SHAP Beeswarm — Direction de l\'Impact', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 🔴 Charger le dataset (obligatoire)
df = pd.read_csv("../data/risk_factors_cervical_cancer.csv")  # <-- mets le bon chemin ici

# 🔹 Séparation features / target
X = df.drop("Biopsy", axis=1)
y = df["Biopsy"]

# 🔹 Split train / test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
import numpy as np
import pandas as pd

# remplacer ?
X = X.replace("?", np.nan)

# convertir en numérique
X = X.apply(pd.to_numeric, errors='coerce')

# imputation
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="mean")
X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

In [ ]:
import numpy as np
import pandas as pd

# 🔹 Nettoyage du dataset (IMPORTANT)
df = df.replace("?", np.nan)
df = df.apply(pd.to_numeric, errors='coerce')

# 🔹 Imputation globale
df = df.fillna(df.mean())

# 🔹 Split
from sklearn.model_selection import train_test_split

X = df.drop("Biopsy", axis=1)
y = df["Biopsy"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 🔹 Maintenant ton code SHAP fonctionne sans problème

In [ ]:
from sklearn.neural_network import MLPClassifier

best_model = MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42)

best_model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = best_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
idx_pos = np.where(y_test.values == 1)[0]
pos = idx_pos[0] if len(idx_pos) > 0 else 0

X_patient = X_test.iloc[[pos]]

prob_patient = best_model.predict_proba(X_patient)[:, 1][0]

print(f'Patient analysé (idx={pos}) — Biopsy réel={y_test.iloc[pos]} | Prob={prob_patient:.4f}')

In [ ]:
import shap

# 🔹 Data sample (important: même format)
X_sample = X_test.iloc[:10]

# 🔹 Explainer
explainer = shap.KernelExplainer(best_model.predict_proba, X_train.sample(100))

# 🔹 SHAP values
shap_values = explainer.shap_values(X_sample)

# 🔹 Cas spécial : vérifier si shap_values est une liste
if isinstance(shap_values, list):
    shap_to_plot = shap_values[1]  # classe 1 (biopsy positif)
else:
    shap_to_plot = shap_values

# 🔹 CORRECTION CRITIQUE : enlever colonne supplémentaire si elle existe
if shap_to_plot.shape[1] > X_sample.shape[1]:
    shap_to_plot = shap_to_plot[:, :X_sample.shape[1]]

# 🔹 Plot
shap.summary_plot(shap_to_plot, X_sample)

In [ ]:
shap.summary_plot(shap_to_plot, X_sample)

In [ ]:
sv = shap_values[1]

# 🔹 Si 3D → prendre la bonne classe
if len(sv.shape) == 3:
    sv = sv[:, :, 1]

# 🔹 Vérification
print("Shape SV after fix:", sv.shape)

# 🔹 IMPORTANT : s'assurer que sv est 2D
sv = sv.reshape(-1, X_test.shape[1])

# 🔹 Calcul des importances
top2 = pd.Series(
    np.abs(sv).mean(axis=0),
    index=X_test.columns
).sort_values(ascending=False).head(2).index.tolist()

print("Top 2 features:", top2)

In [ ]:
try:
    # 🔹 choisir un index valide
    i = min(pos, sv.shape[0] - 1)

    expl = shap.Explanation(
        values=sv[i],
        base_values=ev,
        data=X_test.iloc[i].values,
        feature_names=X_test.columns.tolist()
    )

    plt.figure(figsize=(10, 8))
    shap.plots.waterfall(expl, show=False)
    plt.title(f'Waterfall — Patient {i}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../outputs/shap_waterfall.png', dpi=150, bbox_inches='tight')
    plt.show()

except Exception as e:
    print(f'Waterfall non disponible : {e}')

## 6. Top features SHAP vs Corrélation

In [ ]:
import numpy as np
import pandas as pd

# 🔹 Cas multi-classe / binaire
if isinstance(sv, list):
    shap_values = sv[1]   # classe 1
else:
    shap_values = sv

# 🔹 Vérifier dimensions
print("Shape SHAP:", shap_values.shape)

# 🔹 Si 3D → prendre la classe positive
if shap_values.ndim == 3:
    shap_values = shap_values[:, :, 1]

# 🔹 Moyenne des valeurs absolues
shap_imp = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=X_test.columns
).sort_values(ascending=False)

# 🔹 Affichage TOP 10
print('=== TOP 10 Features les plus importantes (SHAP) ===')
for i, (feat, val) in enumerate(shap_imp.head(10).items()):
    print(f'{i+1:2d}. {feat:<45} {val:.4f}')

## 7. Rapport final complet

In [ ]:
print('\n' + '═'*62)
print('  RAPPORT FINAL DU PROJET ML')
print('  Cervical Cancer Risk Prediction')
print('  Hadil Dhaya · 4th Year Data Science · Group 5')
print('═'*62)

print('\n📂 DATASET')
print('  Source    : UCI ML Repository — Cervical Cancer Risk Factors')
print('  Patients  : 858  |  Features : 36  |  Cible : Biopsy')
print('  Cancer    : ~6% des cas  →  Déséquilibre important')

print('\n🔧 PREPROCESSING')
print('  1. Suppression features > 80% NaN')
print('  2. Imputation KNN (k=5)')
print('  3. Feature Engineering : age_first_sex_gap, smoke_exposure, stds_score')
print('  4. SMOTE — rééquilibrage du train set')
print('  5. StandardScaler — normalisation')

print('\n🤖 MODÈLES TESTÉS (7)')
for m in ['Logistic Regression','K-Nearest Neighbors','Random Forest',
          'SVM (RBF)','Gradient Boosting','AdaBoost','MLP Neural Network']:
    marker = '→ MEILLEUR' if m == best_name else ''
    print(f'  • {m} {marker}')

print(f'\n🏆 MEILLEUR MODÈLE : {best_name}')
print(f'  AUC-ROC   : {auc:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print(f'  Recall    : {rec:.4f}  ← maximisé (cancers manqués = {fn})')
print(f'  Precision : {prec:.4f}')
print(f'  Accuracy  : {acc:.4f}')
print(f'  CV AUC    : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

print('\n🔍 EXPLAINABILITY')
top3 = shap_imp.head(3)
for feat, val in top3.items():
    print(f'  • {feat:<45} SHAP={val:.4f}')

print('\n⚕️  CONCLUSION MÉDICALE')
print('  • Optimiser le Recall > Accuracy : 1 cancer manqué = risque vital')
print('  • Les features IST, âge et tabagisme sont les plus prédictives')
print('  • Le modèle est interprétable grâce à SHAP')
print('  • Usage : outil d\'aide à la décision, PAS de diagnostic autonome')

print('\n📁 FICHIERS GÉNÉRÉS')
for f in sorted(os.listdir('../outputs/')):
    print(f'  ✅ outputs/{f}')

print('\n' + '═'*62)
print('  ✅ PROJET COMPLET — Prêt pour la soutenance !')
print('═'*62)

In [ ]:
import os
print(os.getcwd())

In [ ]:
import joblib

base_path = "C:/Users/msi/Downloads/Project-Machine-Learning/outputs"

imputer = joblib.load(base_path + "/imputer.pkl")
scaler = joblib.load(base_path + "/scaler.pkl")
model = joblib.load(base_path + "/best_model_tuned.pkl")

In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("imputer", imputer),
    ("scaler", scaler),
    ("model", model)
])

joblib.dump(pipeline, os.path.join(base_path, "final_pipeline.pkl"))

print("✅ Pipeline créé !")

In [ ]:
expected_cols = pipeline.feature_names_in_

print("Colonnes attendues :", list(expected_cols))
print("Colonnes actuelles :", list(X_test.columns))

X_test_fixed = X_test[expected_cols]

In [ ]:
missing_cols = set(pipeline.feature_names_in_) - set(X_test.columns)

print("Colonnes manquantes :", missing_cols)

In [ ]:
for col in missing_cols:
    X_test[col] = 0  # valeur par défaut

In [ ]:
X_test_fixed = X_test[pipeline.feature_names_in_]

In [ ]:
df = pd.read_csv("../data/risk_factors_cervical_cancer.csv")  # ou ton df final

In [ ]:
X = df.drop("Biopsy", axis=1)
y = df["Biopsy"]

print(X.columns)

In [ ]:
import pandas as pd

df = pd.read_csv("../data/risk_factors_cervical_cancer.csv")  # ou ton df final

In [ ]:
X = df.drop("Biopsy", axis=1)
y = df["Biopsy"]

print("Nombre de features :", X.shape[1])

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier

pipeline = Pipeline([
    ("imputer", KNNImputer()),
    ("scaler", StandardScaler()),
    ("model", GradientBoostingClassifier())
])

In [ ]:
import numpy as np
df = pd.read_csv("../data/risk_factors_cervical_cancer.csv")  # ou ton df final

df = df.replace("?", np.nan)

In [ ]:
df = df.apply(pd.to_numeric)

In [ ]:
print(df.isnull().sum())

In [ ]:
X = df.drop("Biopsy", axis=1)
y = df["Biopsy"]

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier

pipeline = Pipeline([
    ("imputer", KNNImputer()),
    ("scaler", StandardScaler()),
    ("model", GradientBoostingClassifier())
])

In [ ]:
pipeline.fit(X, y)

In [ ]:
print(pipeline.predict_proba(X.iloc[[0]]))

In [ ]:
import joblib
joblib.dump(pipeline, "../outputs/final_pipeline.pkl")

In [ ]:
print(pipeline.predict_proba(X.iloc[[0]]))

In [ ]:
import pandas as pd

def test_case(data):
    df = pd.DataFrame([data])
    
    # aligner avec le modèle
    df = df.reindex(columns=pipeline.feature_names_in_, fill_value=0)
    
    proba = pipeline.predict_proba(df)[0][1]
    
    if proba < 0.3:
        risk = "🟢 Faible"
    elif proba < 0.7:
        risk = "🟡 Moyen"
    else:
        risk = "🔴 Élevé"
    
    print("Probabilité:", proba)
    print("Risque:", risk)